In [42]:
import json
import pandas as pd

In [43]:
gold_standard_dataset = pd.read_csv('gold_standard_dataset.csv')

contains_fields = gold_standard_dataset["field"].str.contains(
        r"number_of_prior_pregnancies|time_seen_units|time_birth_units|\
        systolic_blood_pressure|diastolic_blood_pressure|rbs_measured|\
        given_bilirubin|primary_admission_diagnosis|secondary_admission_diagnosis",
    regex=True, na=False
    )
gold_standard_dataset = gold_standard_dataset[~contains_fields]
gold_standard_dataset.head()

,hospital,patient_id,form_type,page,field,value
0,76,76000172,NAR,2,appearance,well
1,76,76000172,NAR,2,capillary_refill_in_seconds,2
2,76,76000172,NAR,2,chest_indrawing,TRUE
3,76,76000172,NAR,2,given_bcg,NaN
4,76,76000172,NAR,2,given_bilirubin,NaN


In [46]:
# db.webui_form_processor_stats.findOne({ image_filename: "ITF_40000133_page_1.png" })

gold_standard_dataset[
(gold_standard_dataset["field"]=="pulse_oximetry") & \
(gold_standard_dataset["form_type"]=="ITF") & \
(gold_standard_dataset["patient_id"]==40000133)]

,hospital,patient_id,form_type,page,field,value
6565,40,40000133,ITF,1,pulse_oximetry,69


In [47]:
page_fields = gold_standard_dataset.groupby(["form_type", "page"])['field'].agg(lambda x: sorted(x.unique()))
gold_data_fields = {
    f"{form_type}_{page}": fields
    for (form_type, page), fields in page_fields.items()
}
gold_data_fields

{'ITF_1': ['abnormal_placenta',
  'anc_visits',
  'antenatal_steroids',
  'apgar_10m',
  'apgar_1m',
  'apgar_5m',
  'attended_anc',
  'baby_age',
  'baby_from',
  'birth_date',
  'birth_weight',
  'blood_group',
  'chest_compressions',
  'date_estimated_delivery_date',
  'date_last_menstrual_period',
  'delivery_type',
  'fetal_distress',
  'gestation_in_weeks',
  'given_bcg',
  'given_chlorhexidine',
  'given_teo',
  'given_vitamin_k',
  'gravida',
  'had_cs',
  'has_fever',
  'maternal_status',
  'multiple_pregnancy',
  'mum_age_in_years',
  'mum_had_antepartum_haemorrhage',
  'mum_had_diabetes',
  'mum_had_eclampsia',
  'mum_had_hep_b',
  'mum_had_hypertension_in_pregnancy',
  'mum_had_pre_eclampsia',
  'mum_had_vdrl',
  'mum_has_anc_ultrasound',
  'mum_on_arvs',
  'mum_pmtct_status',
  'mum_treated_for_tb',
  'parity_abortions',
  'parity_live',
  'passed_meconium',
  'placenta_complete',
  'prescribed_antibiotics',
  'prescribed_cpap',
  'prescribed_opv',
  'prescribed_oxygen',
 

In [48]:
with open("mongodb_key_structure.json", "r", encoding="utf-8") as f:
    llm_fields = json.load(f)

llm_fields

{'NAR_2': {'general_examination': ['abdominal_distension_present',
   "air_entry_in_baby's_lungs",
   "baby's_cry",
   "baby's_muscle_tone",
   "baby's_skin_condition",
   'baby_has_crackles',
   'baby_has_grunting',
   'baby_is_irritable',
   'capillary_refill_time_at_sternal_site',
   'condition_of_the_umbilicus',
   "cyanosis_present_in_baby's_central_body",
   'does_baby_have_bulging_fontanelle',
   'general_appearance_of_baby',
   'indrawing_of_lower_chest',
   'level_of_jaundice',
   'pallor_or_anaemia_present_in_baby',
   'presence_of_heart_murmur',
   'retraction_of_intercostal_muscles',
   'retraction_of_xiphoid_process'],
  'further_examination': ['birth_defects_present_in_baby',
   'birth_injury_or_other_abnormalities',
   'cleft_lip_or_palate_present',
   'further_examination_findings_for_respiratory_cardiovascular_git_gu_skin_and_birth_trauma',
   'hydrocephalus_present',
   'limb_abnormalities_present',
   'major_gastrointestinal_abnormality',
   'microcephaly_present',
 

In [12]:
llm_gold_key_mapper = {}
llm_gold_key_mapper["ITF_1"]={
  'abnormal_placenta':'placental_abnormalities',
  'anc_visits':'number_of_anc_visits_attended',
  'antenatal_steroids':'corticosteroids_given',
  'apgar_10m':'apgar_score_at_10_minutes',
  'apgar_1m': 'apgar_score_at_1_minute',
  'apgar_5m':'apgar_score_at_5_minutes',
  'attended_anc':'mother_attended_antenatal_care',
  'baby_age':'neonatal_age',
  'baby_from':'location_baby_originated_from',
  'birth_date':"baby's_date_of_birth",
  'birth_weight':'birth_weight_in_grams',
  'blood_group':"mother's_blood_group",
  'chest_compressions':'chest_compressions_performed',
  'date_estimated_delivery_date':'expected_date_of_delivery',
  'date_last_menstrual_period': 'last_menstrual_period',
  'delivery_type':'mode_of_delivery',
  'fetal_distress':'fetal_distress_during_labour',
  'gestation_in_weeks':'gestational_age_at_delivery_in_weeks',
  'given_bcg':'bcg_vaccine',
  'given_chlorhexidine':'chlorhexidine_cord_care',
  'given_teo': 'thermal_care', # mis-labelled in original dataset
  'given_vitamin_k':'vitamin_k_prophylaxis_given',
  'gravida':'total_number_of_pregnancies',
  'had_cs':'type_of_caesarean_section',
  'has_fever':'maternal_fever_present',
  'maternal_status':"mother's_current_location",
  'multiple_pregnancy':'multiple_pregnancy',
  'mum_age_in_years': "mother's_age_in_years",
  'mum_had_antepartum_haemorrhage':'antepartum_hemorrhage',
  'mum_had_diabetes':'mother_has_diabetes',
  'mum_had_eclampsia':'eclampsia',
  'mum_had_hep_b':'hepatitis_b_vaccine',
  'mum_had_hypertension_in_pregnancy':'hypertension_in_pregnancy',
  'mum_had_pre_eclampsia':'pre-eclampsia',
  'mum_had_vdrl':'syphilis_screening_vdrl_test',
  'mum_has_anc_ultrasound':'antenatal_ultrasound_performed',
  'mum_on_arvs': 'mother_on_antiretroviral_therapy',
  'mum_pmtct_status':'hiv_status_prevention_of_mother-to-child_transmission',
  'mum_treated_for_tb':'mother_on_tb_treatment',
  'parity_abortions':'number_of_stillbirths/deaths',
  'parity_live':'number_of_live_births',
  'pulse_oximetry':'oxygen_saturation',
  'passed_meconium':'meconium-stained_amniotic_fluid',
  'placenta_complete':'placenta_completely_delivered',
  'prescribed_antibiotics':'mother_on_antibiotics',
  'prescribed_cpap':'cpap_support',
  'prescribed_opv':'oral_polio_vaccine',
  'prescribed_oxygen':'supplemental_oxygen',
  'pulse_rate':'heart_rate_beats_per_minute',
  'rapture_of_membrane':'rupture_of_membranes_timing_eg_>18h',
  'respiratory_rate':'respiratory_rate',
  'rhesus':'rhesus_factor_status',
  'sex':"baby's_sex",
  'temparature':'temperature_in_°c',
  'was_resuscitated':'bag_and_mask_ventilation_given',
  'weight':'current_weight_in_grams',
}

54

In [49]:
llm_fields['NAR_1']

{'infant_details': ['apgar_score_at_10_minutes',
  'apgar_score_at_1_minute',
  'apgar_score_at_5_minutes',
  "baby's_age_in_days",
  "baby's_date_of_birth",
  "baby's_sex",
  "baby's_time_of_birth",
  'bag_and_mask_ventilation_given',
  'date_infant_admitted',
  'gestational_age_at_delivery_in_weeks',
  'gestational_age_calculated_from',
  'if_baby_is_born_outside_facility',
  'is_baby_born_outside_facility',
  'mode_of_delivery',
  'multiple_deliveries',
  'number_of_fetuses_in_multiple_pregnancy',
  'rupture_of_membranes_timing_in_hours',
  'time_baby_seen',
  'type_of_caesarean_section'],
 'mother_details': ['antenatal_ultrasound_performed',
  'antepartum_hemorrhage',
  'expected_date_of_delivery',
  'hiv_status_prevention_of_mother-to-child_transmission',
  'hypertension_in_pregnancy',
  "mother's_age_in_years",
  "mother's_blood_group",
  'mother_given_hbig_treatment',
  'mother_had_hepatitis_b',
  'mother_had_prolonged_labour',
  'mother_has_diabetes',
  'mother_on_arvs',
  'num

In [33]:
llm_gold_key_mapper["NAR_1"]={
    'date':'date_infant_admitted',
    'anc_visits':'number_of_anc_visits_attended',
    'apgar_10m':'apgar_score_at_10_minutes',
    'apgar_1m':'apgar_score_at_1_minute',
    'apgar_5m':'apgar_score_at_5_minutes',
    'birth_date':"baby's_date_of_birth",
    'birth_weight':'birth_weight_in_grams',
    'blood_group':"mother's_blood_group",

    'baby_age_in_days':"baby's_age_in_days",
    'born_before_arrival':'is_baby_born_outside_facility',
    'mum_given_HBIG_treatment':'mother_given_hbig_treatment',
    'mum_on_arvs':'mother_on_arvs',
    'head_circumference':'head_circumference_in_cm',
    'length':'length_in_cm',
    'parity_abortions':'number_of_stillbirths/deaths',
    'parity_live':'number_of_live_births',
    'mum_had_hepatitis_b':'mother_had_hepatitis_b',
    'born_where':'if_baby_is_born_outside_facility',
    'date_estimated_delivery_date':'expected_date_of_delivery',
    'delivery_type':'mode_of_delivery',
    'gestation_in_weeks':'gestational_age_at_delivery_in_weeks',
    'gestation_type':'gestational_age_calculated_from',
    'given_anti_D_medication':'rhesus_anti-d_given',
    'had_cs':'type_of_caesarean_section',
    'has_apnoea':'baby_has_apnoea',
    'has_convulsions':'baby_has_convulsions',
    'has_diarhoea':'baby_has_bloody_stool',
    'has_difficulty_breathing':'baby_has_difficulty_breathing',
    'has_difficulty_feeding':'baby_has_difficulty_feeding',
    'has_fever':'baby_has_fever_present',
    'has_vomiting':'baby_has_bilious_vomiting',
    'is_floppy':'baby_is_floppy',
    'is_multiple_delivery':'multiple_deliveries',
    'multiple_delivery_num':'number_of_fetuses_in_multiple_pregnancy',
    'mum_age_in_years': "mother's_age_in_years",
    'mum_had_antepartum_haemorrhage':'antepartum_hemorrhage',
    'mum_had_diabetes':'mother_has_diabetes',
    'mum_had_hypertension_in_pregnancy':'hypertension_in_pregnancy',
    'mum_had_vdrl':'syphilis_screening_vdrl_test',
    'mum_has_anc_ultrasound':'antenatal_ultrasound_performed',
    'mum_pmtct_status':'hiv_status_prevention_of_mother-to-child_transmission',
    'passed_meconium':'baby_passed_meconium_stool',
    'passed_urine':'baby_passed_urine',
    'prolonged_labour':'mother_had_prolonged_labour',
    'pulse_oximetry':'oxygen_saturation',
    'pulse_rate':'heart_rate_beats_per_minute',
    'rapture_of_membrane':'rupture_of_membranes_timing_in_hours',
    'respiratory_rate':'respiratory_rate',
    'rhesus':'rhesus_factor_status',
    'sex':"baby's_sex",
    'temparature':'temperature_in_°c',
    'time_birth':"baby's_time_of_birth",
    'time_seen':'time_baby_seen',
    'was_resuscitated':'bag_and_mask_ventilation_given',
    'weight':'current_weight_in_grams'
}

'{"date": "date_infant_admitted", "anc_visits": "number_of_anc_visits_attended", "apgar_10m": "apgar_score_at_10_minutes", "apgar_1m": "apgar_score_at_1_minute", "apgar_5m": "apgar_score_at_5_minutes", "birth_date": "baby\'s_date_of_birth", "birth_weight": "birth_weight_in_grams", "blood_group": "mother\'s_blood_group", "born_where": "if_baby_is_born_outside_facility", "date_estimated_delivery_date": "expected_date_of_delivery", "delivery_type": "mode_of_delivery", "gestation_in_weeks": "gestational_age_at_delivery_in_weeks", "gestation_type": "gestational_age_calculated_from", "given_anti_D_medication": "rhesus_anti-d_given", "had_cs": "type_of_caesarean_section", "has_apnoea": "baby_has_apnoea", "has_convulsions": "baby_has_convulsions", "has_diarhoea": "baby_has_bloody_stool", "has_difficulty_breathing": "baby_has_difficulty_breathing", "has_difficulty_feeding": "baby_has_difficulty_feeding", "has_fever": "baby_has_fever_present", "has_vomiting": "baby_has_bilious_vomiting", "is_flo

In [38]:
llm_gold_key_mapper["NAR_2"]={
    'appearance':'general_appearance_of_baby',
    'capillary_refill_in_seconds':'capillary_refill_time_at_sternal_site',
    'chest_indrawing':'indrawing_of_lower_chest',
    'cry':"baby's_cry",
    'given_bcg':'bcg_vaccination_given',
    'given_chlorhexidine':'chlorhexidine_given_for_cord_care',
    'given_prophylaxis_pmtct':'prophylaxis_for_prevention_of_mother-to-child_transmission',
    'given_vitamin_k':'vitamin_k_and_topical_eye_ointment_given',
    'has_birth_defects':'birth_defects_present_in_baby',
    'has_bulging_fontanelle':'does_baby_have_bulging_fontanelle',
    'has_central_cyanosis':"cyanosis_present_in_baby's_central_body",
    'has_crackles':'baby_has_crackles',
    'has_good_air_entry':"air_entry_in_baby's_lungs",
    'has_grunting':'baby_has_grunting',
    'has_murmur':'presence_of_heart_murmur',
    'intercostal_retraction':'retraction_of_intercostal_muscles',
    'is_distended':'abdominal_distension_present',
    'is_irritable':'baby_is_irritable',
    'jaundice':'level_of_jaundice',
    'pallor':'pallor_or_anaemia_present_in_baby',
    'prescribed_antibiotics':'antibiotic_therapy_given',
    'prescribed_caffeine_citrate':'caffeine_citrate_given_for_apnoea',
    'prescribed_cpap':'cpap_therapy_given',
    'prescribed_feeds':'feeding/nutrition_support',
    'prescribed_incubator':'incubator_or_warm_environment_provided',
    'prescribed_iv_fluids':'intravenous_fluids_given',
    'prescribed_kmc':'kangaroo_mother_care_provided',
    'prescribed_opv':'opv_polio_vaccination_given',
    'prescribed_oxygen':'oxygen_therapy_given',
    'prescribed_phototherapy':'phototherapy_given_for_jaundice',
    'prescribed_surfactant':'surfactant_therapy_given',
    'prescribed_transfusion':'blood_transfusion_given',
    'skin':"baby's_skin_condition",
    'tone':"baby's_muscle_tone",
    'umbilicus':'condition_of_the_umbilicus',
    'xiphoid_retraction':'retraction_of_xiphoid_process'
}